# Cross-Model Feature Comparison

| Step    | Analysis                                | What we learn                                           |
| ------- | --------------------------------------- | ------------------------------------------------------- |
| **24A** | Standardize feature-importance measures | Put coefficients/importances on a comparable 0–1 scale  |
| **24B** | Create cross-model ranking table        | See each feature's rank under Logistic, DT, RF, XGBoost |
| **24C** | Measure agreement/stability             | Find predictors repeatedly appearing among top features |
| **24D** | Final visualization + interpretation    | Produce the evidence for the report                     |


### Load data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

In [2]:
DATA_DIR = Path("data")

X_train = pd.read_csv(
    DATA_DIR / "(experiment) X_train.csv",
    index_col=0,
    parse_dates=True
)

y_train = pd.read_csv(
    DATA_DIR / "(experiment) y_train.csv",
    index_col=0,
    parse_dates=True
)

X_test = pd.read_csv(
    DATA_DIR / "(experiment) X_test.csv",
    index_col=0,
    parse_dates=True
)

y_test = pd.read_csv(
    DATA_DIR / "(experiment) y_test.csv",
    index_col=0,
    parse_dates=True
)

In [3]:
print("=" * 50)
print("TRAIN / TEST SPLIT")
print("=" * 50)

print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))

print("\nTraining period:")
print(X_train.index.min(), "to", X_train.index.max())

print("\nTesting period:")
print(X_test.index.min(), "to", X_test.index.max())

print("\nTraining class distribution:")
print(y_train.value_counts().sort_index())

print("\nTesting class distribution:")
print(y_test.value_counts().sort_index())

TRAIN / TEST SPLIT
Training observations: 59
Testing observations: 18

Training period:
2007-06-30 00:00:00 to 2021-12-31 00:00:00

Testing period:
2022-03-31 00:00:00 to 2026-06-30 00:00:00

Training class distribution:
PCE_Regime
0             29
1             30
Name: count, dtype: int64

Testing class distribution:
PCE_Regime
0              8
1             10
Name: count, dtype: int64


## 24A — Build the comparison table

In [6]:
# ============================================================
# STEP 24A — BUILD AND TRAIN FOUR CLASSIFICATION MODELS
# ============================================================

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from xgboost import XGBClassifier


# ============================================================
# 1. MAKE SURE y IS 1-DIMENSIONAL
# ============================================================

y_train_1d = np.ravel(y_train)
y_test_1d = np.ravel(y_test)


print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

print("y_train shape:", y_train_1d.shape)
print("y_test shape:", y_test_1d.shape)

Training shape: (59, 73)
Testing shape: (18, 73)
y_train shape: (59,)
y_test shape: (18,)


In [7]:
# ============================================================
# 2. LOGISTIC REGRESSION PIPELINE
# ============================================================

logistic_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])


# ============================================================
# 3. DECISION TREE
# ============================================================

dt_model = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=3,
    random_state=42
)


# ============================================================
# 4. RANDOM FOREST
# ============================================================

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=4,
    min_samples_leaf=2,
    random_state=42
)


# ============================================================
# 5. XGBOOST
# ============================================================

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

In [8]:
# ============================================================
# 6. TRAIN ALL MODELS
# ============================================================

models = {
    "Logistic Regression": logistic_pipeline,
    "Decision Tree": dt_model,
    "Random Forest": rf_model,
    "XGBoost": xgb_model
}


for model_name, model in models.items():

    print(f"Training: {model_name}")

    model.fit(
        X_train,
        y_train_1d
    )


print("\nAll models trained successfully.")

Training: Logistic Regression
Training: Decision Tree
Training: Random Forest
Training: XGBoost

All models trained successfully.


In [9]:
print(logistic_pipeline)
print(dt_model)
print(rf_model)
print(xgb_model)

Pipeline(steps=[('scaler', StandardScaler()),
                ('classifier',
                 LogisticRegression(max_iter=2000, random_state=42))])
DecisionTreeClassifier(max_depth=3, min_samples_leaf=3, random_state=42)
RandomForestClassifier(max_depth=4, min_samples_leaf=2, n_estimators=300,
                       random_state=42)
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,


## 24B — MODEL PERFORMANCE COMPARISON

In [10]:
# ============================================================
# STEP 24B — MODEL PERFORMANCE COMPARISON
# ============================================================

performance_results = []


for model_name, model in models.items():

    # Predictions
    y_pred = model.predict(X_test)

    # Predicted probability for class 1
    y_prob = model.predict_proba(X_test)[:, 1]

    performance_results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test_1d, y_pred),
        "Precision": precision_score(
            y_test_1d,
            y_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test_1d,
            y_pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_test_1d,
            y_pred,
            zero_division=0
        ),
        "ROC_AUC": roc_auc_score(
            y_test_1d,
            y_prob
        )
    })


performance_df = pd.DataFrame(performance_results)

display(
    performance_df.style.format({
        "Accuracy": "{:.3f}",
        "Precision": "{:.3f}",
        "Recall": "{:.3f}",
        "F1": "{:.3f}",
        "ROC_AUC": "{:.3f}"
    })
)

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Logistic Regression,0.556,0.667,0.400,0.500,0.550
1,Decision Tree,0.556,0.556,1.000,0.714,0.488
2,Random Forest,0.556,0.583,0.700,0.636,0.462
3,XGBoost,0.556,0.667,0.400,0.500,0.600


## 24C — CROSS-MODEL FEATURE IMPORTANCE COMPARISON

In [11]:
# ============================================================
# STEP 24C — CROSS-MODEL FEATURE IMPORTANCE COMPARISON
# ============================================================

feature_names = X_train.columns


comparison_df = pd.DataFrame({
    "Feature": feature_names
})


# ============================================================
# LOGISTIC REGRESSION
# ============================================================

logistic_classifier = (
    logistic_pipeline
    .named_steps["classifier"]
)

log_coef = (
    logistic_classifier
    .coef_
    .ravel()
)


comparison_df["Logistic_Coefficient"] = log_coef

comparison_df["Logistic_Importance"] = (
    np.abs(log_coef)
)


# ============================================================
# TREE-BASED MODELS
# ============================================================

tree_models = {
    "DecisionTree": dt_model,
    "RandomForest": rf_model,
    "XGBoost": xgb_model
}


for model_name, model in tree_models.items():

    comparison_df[
        f"{model_name}_Importance"
    ] = model.feature_importances_


display(comparison_df)

,Feature,Logistic_Coefficient,Logistic_Importance,DecisionTree_Importance,RandomForest_Importance,XGBoost_Importance
0,Disposable Personal Income,0.017654,0.017654,0.000000,0.014979,0.020149
1,Personal Income,0.050585,0.050585,0.000000,0.014742,0.000000
2,Real Disposable Income,0.036314,0.036314,0.000000,0.007600,0.000000
3,Average Hourly Earnings,-0.002579,0.002579,0.000000,0.004057,0.000000
4,Weekly Earnings,-0.373120,0.373120,0.000000,0.006841,0.002985
...,...,...,...,...,...,...
68,Tax Refunds,0.449137,0.449137,0.000000,0.017781,0.008106
69,Unemployment Benefits,0.133838,0.133838,0.000000,0.026265,0.025109
70,Oil Price,0.037180,0.037180,0.000000,0.009600,0.010680
71,Gasoline Price,0.000615,0.000615,0.000000,0.019905,0.016414


## 24D — NORMALIZED FEATURE IMPORTANCE

In [12]:
# ============================================================
# STEP 24D — NORMALIZED FEATURE IMPORTANCE
# ============================================================

importance_columns = [
    "Logistic_Importance",
    "DecisionTree_Importance",
    "RandomForest_Importance",
    "XGBoost_Importance"
]


for col in importance_columns:

    total = comparison_df[col].sum()

    if total > 0:

        comparison_df[
            f"{col}_Normalized"
        ] = comparison_df[col] / total

## 24E — CONSENSUS FEATURE IMPORTANCE

In [13]:
# ============================================================
# STEP 24E — CONSENSUS FEATURE IMPORTANCE
# ============================================================

normalized_columns = [
    "Logistic_Importance_Normalized",
    "DecisionTree_Importance_Normalized",
    "RandomForest_Importance_Normalized",
    "XGBoost_Importance_Normalized"
]


comparison_df["Mean_Importance"] = (
    comparison_df[
        normalized_columns
    ].mean(axis=1)
)


comparison_df["Consensus_Rank"] = (
    comparison_df["Mean_Importance"]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)


comparison_ranked = (
    comparison_df
    .sort_values(
        "Mean_Importance",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    comparison_ranked[[
        "Feature",
        "Logistic_Coefficient",
        "Logistic_Importance_Normalized",
        "DecisionTree_Importance_Normalized",
        "RandomForest_Importance_Normalized",
        "XGBoost_Importance_Normalized",
        "Mean_Importance",
        "Consensus_Rank"
    ]]
)

,Feature,Logistic_Coefficient,Logistic_Importance_Normalized,DecisionTree_Importance_Normalized,RandomForest_Importance_Normalized,XGBoost_Importance_Normalized,Mean_Importance,Consensus_Rank
0,Federal Funds Rate,0.076330,0.005127,0.474423,0.034076,0.020406,0.133508,1
1,Exchange Rate,-0.213711,0.014354,0.374166,0.020636,0.038534,0.111922,2
2,Interest Income,-0.256747,0.017244,0.151411,0.010306,0.014010,0.048243,3
3,Rental Income,-0.128276,0.008616,0.000000,0.013644,0.080958,0.025804,4
4,Delinquency Rate,0.257721,0.017310,0.000000,0.029091,0.056557,0.025739,5
...,...,...,...,...,...,...,...,...
68,Housing Starts,-0.021050,0.001414,0.000000,0.006679,0.000000,0.002023,69
69,Core PCE,-0.049145,0.003301,0.000000,0.004411,0.000000,0.001928,70
70,Shelter Inflation,-0.065214,0.004380,0.000000,0.003257,0.000000,0.001909,71
71,PCE Inflation,-0.026104,0.001753,0.000000,0.004917,0.000000,0.001668,72


## 24F — Cross-model rank stability

In [14]:
# ============================================================
# STEP 24F — CROSS-MODEL FEATURE RANK STABILITY
# ============================================================

importance_cols = {
    "Logistic": "Logistic_Importance_Normalized",
    "DecisionTree": "DecisionTree_Importance_Normalized",
    "RandomForest": "RandomForest_Importance_Normalized",
    "XGBoost": "XGBoost_Importance_Normalized"
}


# ------------------------------------------------------------
# Create rank for each model
# Rank 1 = most important
# ------------------------------------------------------------

rank_columns = []

for model_name, col in importance_cols.items():

    rank_col = f"{model_name}_Rank"

    comparison_df[rank_col] = (
        comparison_df[col]
        .rank(
            ascending=False,
            method="min"
        )
    )

    rank_columns.append(rank_col)


# ------------------------------------------------------------
# Mean rank
# Lower = more consistently important
# ------------------------------------------------------------

comparison_df["Mean_Rank"] = (
    comparison_df[rank_columns]
    .mean(axis=1)
)


# ------------------------------------------------------------
# Rank variability
# Lower = more stable across models
# ------------------------------------------------------------

comparison_df["Rank_Std"] = (
    comparison_df[rank_columns]
    .std(axis=1)
)


# ------------------------------------------------------------
# Final table
# ------------------------------------------------------------

stability_table = (
    comparison_df[[
        "Feature",
        "Logistic_Rank",
        "DecisionTree_Rank",
        "RandomForest_Rank",
        "XGBoost_Rank",
        "Mean_Rank",
        "Rank_Std"
    ]]
    .sort_values(
        ["Mean_Rank", "Rank_Std"],
        ascending=[True, True]
    )
    .reset_index(drop=True)
)


display(stability_table.head(20))

,Feature,Logistic_Rank,DecisionTree_Rank,RandomForest_Rank,XGBoost_Rank,Mean_Rank,Rank_Std
0,Delinquency Rate,23.0,4.0,6.0,2.0,8.75,9.639329
1,Dividend Income,3.0,4.0,26.0,4.0,9.25,11.176612
2,Corporate Bond Yield,16.0,4.0,4.0,15.0,9.75,6.652067
3,Credit Card & Revolving Loan Growth,17.0,4.0,5.0,14.0,10.00,6.480741
4,Employment Rate,26.0,4.0,3.0,9.0,10.50,10.661457
5,Savings Rate,1.0,4.0,11.0,27.0,10.75,11.615363
6,Economic Policy Uncertainty,2.0,4.0,22.0,16.0,11.00,9.591663
7,Exchange Rate,31.0,2.0,10.0,3.0,11.50,13.478378
8,Labor Force Participation Rate,21.0,4.0,2.0,21.0,12.00,10.424331
9,Personal Loan Rate,10.0,4.0,9.0,30.0,13.25,11.470978


## 24G — TOP-10 MODEL AGREEMENT

#### How many models identify each feature among their Top 10?

In [15]:
# ============================================================
# STEP 24G — TOP-10 MODEL AGREEMENT
# ============================================================

TOP_N = 10


comparison_df["Top10_Count"] = sum(
    comparison_df[col] <= TOP_N
    for col in rank_columns
)


comparison_df["Top10_Percent"] = (
    comparison_df["Top10_Count"]
    / len(rank_columns)
)


agreement_table = (
    comparison_df[[
        "Feature",
        "Logistic_Rank",
        "DecisionTree_Rank",
        "RandomForest_Rank",
        "XGBoost_Rank",
        "Mean_Rank",
        "Rank_Std",
        "Top10_Count",
        "Top10_Percent"
    ]]
    .sort_values(
        ["Top10_Count", "Mean_Rank"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)


display(agreement_table.head(20))

,Feature,Logistic_Rank,DecisionTree_Rank,RandomForest_Rank,XGBoost_Rank,Mean_Rank,Rank_Std,Top10_Count,Top10_Percent
0,Delinquency Rate,23.0,4.0,6.0,2.0,8.75,9.639329,3,0.75
1,Dividend Income,3.0,4.0,26.0,4.0,9.25,11.176612,3,0.75
2,Employment Rate,26.0,4.0,3.0,9.0,10.50,10.661457,3,0.75
3,Exchange Rate,31.0,2.0,10.0,3.0,11.50,13.478378,3,0.75
4,Personal Loan Rate,10.0,4.0,9.0,30.0,13.25,11.470978,3,0.75
5,Vehicle Sales,8.0,4.0,40.0,6.0,14.50,17.078251,3,0.75
6,Full-time Employment,6.0,4.0,43.0,10.0,15.75,18.337121,3,0.75
7,Corporate Bond Yield,16.0,4.0,4.0,15.0,9.75,6.652067,2,0.50
8,Credit Card & Revolving Loan Growth,17.0,4.0,5.0,14.0,10.00,6.480741,2,0.50
9,Savings Rate,1.0,4.0,11.0,27.0,10.75,11.615363,2,0.50


In [16]:
# ============================================================
# STEP 24G — CORRECTED TOP-N MODEL AGREEMENT
# Zero-importance features do NOT count as Top-N
# ============================================================

TOP_N = 10


model_info = {
    "Logistic": (
        "Logistic_Importance_Normalized",
        "Logistic_Rank"
    ),

    "DecisionTree": (
        "DecisionTree_Importance_Normalized",
        "DecisionTree_Rank"
    ),

    "RandomForest": (
        "RandomForest_Importance_Normalized",
        "RandomForest_Rank"
    ),

    "XGBoost": (
        "XGBoost_Importance_Normalized",
        "XGBoost_Rank"
    )
}


# ------------------------------------------------------------
# Create TRUE Top-10 indicator
# Must satisfy BOTH:
#   1. rank <= 10
#   2. importance > 0
# ------------------------------------------------------------

top10_columns = []


for model_name, (importance_col, rank_col) in model_info.items():

    indicator_col = f"{model_name}_Top10"

    comparison_df[indicator_col] = (
        (comparison_df[rank_col] <= TOP_N)
        &
        (comparison_df[importance_col] > 0)
    ).astype(int)

    top10_columns.append(indicator_col)


# ------------------------------------------------------------
# Count model agreement
# ------------------------------------------------------------

comparison_df["Top10_Count"] = (
    comparison_df[top10_columns]
    .sum(axis=1)
)


comparison_df["Top10_Percent"] = (
    comparison_df["Top10_Count"]
    / len(model_info)
)


# ------------------------------------------------------------
# Final agreement table
# ------------------------------------------------------------

agreement_table_corrected = (
    comparison_df[[
        "Feature",

        "Logistic_Rank",
        "DecisionTree_Rank",
        "RandomForest_Rank",
        "XGBoost_Rank",

        "Logistic_Top10",
        "DecisionTree_Top10",
        "RandomForest_Top10",
        "XGBoost_Top10",

        "Mean_Rank",
        "Rank_Std",

        "Top10_Count",
        "Top10_Percent"
    ]]
    .sort_values(
        ["Top10_Count", "Mean_Rank"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)


display(
    agreement_table_corrected.head(20)
)

,Feature,Logistic_Rank,DecisionTree_Rank,RandomForest_Rank,XGBoost_Rank,Logistic_Top10,DecisionTree_Top10,RandomForest_Top10,XGBoost_Top10,Mean_Rank,Rank_Std,Top10_Count,Top10_Percent
0,Exchange Rate,31.0,2.0,10.0,3.0,0,1,1,1,11.50,13.478378,3,0.75
1,Delinquency Rate,23.0,4.0,6.0,2.0,0,0,1,1,8.75,9.639329,2,0.50
2,Dividend Income,3.0,4.0,26.0,4.0,1,0,0,1,9.25,11.176612,2,0.50
3,Employment Rate,26.0,4.0,3.0,9.0,0,0,1,1,10.50,10.661457,2,0.50
4,Personal Loan Rate,10.0,4.0,9.0,30.0,1,0,1,0,13.25,11.470978,2,0.50
5,Vehicle Sales,8.0,4.0,40.0,6.0,1,0,0,1,14.50,17.078251,2,0.50
6,Full-time Employment,6.0,4.0,43.0,10.0,1,0,0,1,15.75,18.337121,2,0.50
7,Federal Funds Rate,51.0,1.0,1.0,17.0,0,1,1,0,17.50,23.572583,2,0.50
8,Corporate Bond Yield,16.0,4.0,4.0,15.0,0,0,1,0,9.75,6.652067,1,0.25
9,Credit Card & Revolving Loan Growth,17.0,4.0,5.0,14.0,0,0,1,0,10.00,6.480741,1,0.25


## Interpretation: 

The relationship between Exchange Rate and PCE growth regimes may be nonlinear, conditional, or interactive rather than a simple linear relationship.

- Tier 1 — Strongest cross-model consensus

Exchange Rate — 3/4 models (75%)\
This is currently the only feature satisfying this level.

- Tier 2 — Moderate cross-model evidence
   
Seven features appear in the Top 10 of two models:\
1. Delinquency Rate
2. Dividend Income
3. Employment Rate
4. Personal Loan Rate
5. Vehicle Sales
6. Full-time Employment
7. Federal Funds Rate

These deserve attention, but their importance is more model-dependent.

- Tier 3 — Model-specific signals
  
Features with only 1/4 agreement include Savings Rate, Economic Policy Uncertainty, Labor Force Participation Rate, Personal Savings, Inflation Expectations, Mortgage Rate, House Price Index, Rental Income, etc.


# Step 25 — Temporal Feature Stability with Walk-Forward Validation


- Step 24 asked:\
Do different ML algorithms identify the same important features?
- Step 25 asks:\
Do those features remain important across different historical periods?

## 25A — Reconstruct the full chronological dataset


In [17]:
# ============================================================
# STEP 25A — RECONSTRUCT FULL CHRONOLOGICAL DATASET
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Combine X
# ------------------------------------------------------------

X_full = pd.concat([
    X_train,
    X_test
]).sort_index()


# ------------------------------------------------------------
# Combine y
# ------------------------------------------------------------

y_train_series = pd.Series(
    np.ravel(y_train),
    index=X_train.index,
    name="PCE_Regime"
)

y_test_series = pd.Series(
    np.ravel(y_test),
    index=X_test.index,
    name="PCE_Regime"
)


y_full = pd.concat([
    y_train_series,
    y_test_series
]).sort_index()


# ------------------------------------------------------------
# Check
# ------------------------------------------------------------

print("=" * 60)
print("FULL DATASET")
print("=" * 60)

print("Observations:", len(X_full))
print("Features:", X_full.shape[1])

print("\nPeriod:")
print(X_full.index.min(), "to", X_full.index.max())

print("\nClass distribution:")
print(y_full.value_counts().sort_index())

print("\nX/y aligned:")
print(X_full.index.equals(y_full.index))

FULL DATASET
Observations: 77
Features: 73

Period:
2007-06-30 00:00:00 to 2026-06-30 00:00:00

Class distribution:
PCE_Regime
0    37
1    40
Name: count, dtype: int64

X/y aligned:
True


## 25B — Define expanding walk-forward folds

In [18]:
# ============================================================
# STEP 25B — DEFINE WALK-FORWARD FOLDS
# ============================================================

test_years = list(range(2018, 2026))

folds = []


for test_year in test_years:

    train_mask = X_full.index.year < test_year
    test_mask = X_full.index.year == test_year

    X_fold_train = X_full.loc[train_mask]
    X_fold_test = X_full.loc[test_mask]

    y_fold_train = y_full.loc[train_mask]
    y_fold_test = y_full.loc[test_mask]

    folds.append({
        "Test_Year": test_year,

        "X_train": X_fold_train,
        "X_test": X_fold_test,

        "y_train": y_fold_train,
        "y_test": y_fold_test
    })


# ------------------------------------------------------------
# Inspect folds
# ------------------------------------------------------------

fold_summary = []

for fold in folds:

    fold_summary.append({
        "Test_Year": fold["Test_Year"],

        "Train_Start":
            fold["X_train"].index.min(),

        "Train_End":
            fold["X_train"].index.max(),

        "Train_N":
            len(fold["X_train"]),

        "Test_N":
            len(fold["X_test"])
    })


fold_summary_df = pd.DataFrame(fold_summary)

display(fold_summary_df)

,Test_Year,Train_Start,Train_End,Train_N,Test_N
0,2018,2007-06-30,2017-12-31,43,4
1,2019,2007-06-30,2018-12-31,47,4
2,2020,2007-06-30,2019-12-31,51,4
3,2021,2007-06-30,2020-12-31,55,4
4,2022,2007-06-30,2021-12-31,59,4
5,2023,2007-06-30,2022-12-31,63,4
6,2024,2007-06-30,2023-12-31,67,4
7,2025,2007-06-30,2024-12-31,71,4


## 25C — Define fresh models for every fold

In [19]:
# ============================================================
# STEP 25C — MODEL FACTORY
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier


def create_models():

    models = {

        "Logistic": Pipeline([
            (
                "scaler",
                StandardScaler()
            ),
            (
                "classifier",
                LogisticRegression(
                    max_iter=2000,
                    random_state=42
                )
            )
        ]),

        "DecisionTree":
            DecisionTreeClassifier(
                max_depth=3,
                min_samples_leaf=3,
                random_state=42
            ),

        "RandomForest":
            RandomForestClassifier(
                n_estimators=300,
                max_depth=4,
                min_samples_leaf=2,
                random_state=42
            ),

        "XGBoost":
            XGBClassifier(
                n_estimators=100,
                max_depth=3,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                eval_metric="logloss"
            )
    }

    return models

In [20]:
models = create_models()

## 25D — Walk-forward training + feature extraction

For every year:
1. Train four models.
2. Predict the following year.
3. measure performance.
4. Extract feature importance.
5. Rank features.
6. Record whether each feature is Top 10.

In [21]:
# ============================================================
# STEP 25D — WALK-FORWARD MODELING
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)


performance_records = []
importance_records = []


for fold in folds:

    test_year = fold["Test_Year"]

    X_tr = fold["X_train"]
    X_te = fold["X_test"]

    y_tr = np.ravel(fold["y_train"])
    y_te = np.ravel(fold["y_test"])


    print("=" * 60)
    print(f"TEST YEAR: {test_year}")
    print("=" * 60)

    print(
        f"Train: {X_tr.index.min().date()} "
        f"to {X_tr.index.max().date()}"
    )

    print(f"Test observations: {len(X_te)}")


    # --------------------------------------------------------
    # Create NEW models for this fold
    # --------------------------------------------------------

    models = create_models()


    # --------------------------------------------------------
    # Train each model
    # --------------------------------------------------------

    for model_name, model in models.items():

        model.fit(X_tr, y_tr)


        # ====================================================
        # PERFORMANCE
        # ====================================================

        y_pred = model.predict(X_te)

        y_prob = model.predict_proba(X_te)[:, 1]


        # ROC-AUC requires both classes in test year
        if len(np.unique(y_te)) == 2:

            auc = roc_auc_score(
                y_te,
                y_prob
            )

        else:

            auc = np.nan


        performance_records.append({

            "Test_Year": test_year,

            "Model": model_name,

            "Accuracy":
                accuracy_score(
                    y_te,
                    y_pred
                ),

            "Precision":
                precision_score(
                    y_te,
                    y_pred,
                    zero_division=0
                ),

            "Recall":
                recall_score(
                    y_te,
                    y_pred,
                    zero_division=0
                ),

            "F1":
                f1_score(
                    y_te,
                    y_pred,
                    zero_division=0
                ),

            "ROC_AUC": auc
        })


        # ====================================================
        # FEATURE IMPORTANCE
        # ====================================================

        if model_name == "Logistic":

            classifier = (
                model.named_steps["classifier"]
            )

            importance = np.abs(
                classifier.coef_.ravel()
            )

        else:

            importance = (
                model.feature_importances_
            )


        # ----------------------------------------------------
        # Normalize
        # ----------------------------------------------------

        total = importance.sum()

        if total > 0:

            normalized_importance = (
                importance / total
            )

        else:

            normalized_importance = importance


        # ----------------------------------------------------
        # Save each feature
        # ----------------------------------------------------

        for feature, imp in zip(
            X_full.columns,
            normalized_importance
        ):

            importance_records.append({

                "Test_Year":
                    test_year,

                "Model":
                    model_name,

                "Feature":
                    feature,

                "Importance":
                    imp
            })

TEST YEAR: 2018
Train: 2007-06-30 to 2017-12-31
Test observations: 4
TEST YEAR: 2019
Train: 2007-06-30 to 2018-12-31
Test observations: 4
TEST YEAR: 2020
Train: 2007-06-30 to 2019-12-31
Test observations: 4
TEST YEAR: 2021
Train: 2007-06-30 to 2020-12-31
Test observations: 4
TEST YEAR: 2022
Train: 2007-06-30 to 2021-12-31
Test observations: 4
TEST YEAR: 2023
Train: 2007-06-30 to 2022-12-31
Test observations: 4
TEST YEAR: 2024
Train: 2007-06-30 to 2023-12-31
Test observations: 4
TEST YEAR: 2025
Train: 2007-06-30 to 2024-12-31
Test observations: 4


## 25E — Create results DataFrames

In [22]:
# ============================================================
# STEP 25E — RESULTS TABLES
# ============================================================

walkforward_performance = pd.DataFrame(
    performance_records
)

walkforward_importance = pd.DataFrame(
    importance_records
)


print("Performance records:")
display(walkforward_performance.head(12))


print("\nFeature importance records:")
display(walkforward_importance.head(12))

Performance records:


,Test_Year,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,2018,Logistic,0.25,0.250000,1.0,0.400000,0.000000
1,2018,DecisionTree,0.75,0.000000,0.0,0.000000,0.500000
2,2018,RandomForest,0.25,0.250000,1.0,0.400000,0.333333
3,2018,XGBoost,0.25,0.250000,1.0,0.400000,0.000000
4,2019,Logistic,0.25,0.000000,0.0,0.000000,0.666667
5,2019,DecisionTree,0.25,0.000000,0.0,0.000000,0.500000
6,2019,RandomForest,0.25,0.000000,0.0,0.000000,1.000000
7,2019,XGBoost,0.25,0.000000,0.0,0.000000,1.000000
8,2020,Logistic,0.50,0.500000,1.0,0.666667,1.000000
9,2020,DecisionTree,1.00,1.000000,1.0,1.000000,1.000000



Feature importance records:


,Test_Year,Model,Feature,Importance
0,2018,Logistic,Disposable Personal Income,0.000634
1,2018,Logistic,Personal Income,0.001348
2,2018,Logistic,Real Disposable Income,0.000814
3,2018,Logistic,Average Hourly Earnings,0.000083
4,2018,Logistic,Weekly Earnings,0.028028
5,2018,Logistic,Wage Growth,0.007096
6,2018,Logistic,Labor Compensation,0.014133
7,2018,Logistic,Self-employment Income,0.000115
8,2018,Logistic,Rental Income,0.000600
9,2018,Logistic,Dividend Income,0.029469


## 25F — Rank features within each fold/model

In [24]:
# ============================================================
# STEP 25F — TEMPORAL FEATURE RANKING
# ============================================================

walkforward_importance["Rank"] = (
    walkforward_importance
    .groupby([
        "Test_Year",
        "Model"
    ])["Importance"]
    .rank(
        ascending=False,
        method="min"
    )
)


# ------------------------------------------------------------
# TRUE Top-10
#
# Must have:
# Rank <= 10
# AND
# Importance > 0
# ------------------------------------------------------------

walkforward_importance["Top10"] = (
    (
        walkforward_importance["Rank"] <= 10
    )
    &
    (
        walkforward_importance["Importance"] > 0
    )
).astype(int)


display(
    walkforward_importance.head(20)
)

,Test_Year,Model,Feature,Importance,Rank,Top10
0,2018,Logistic,Disposable Personal Income,0.000634,66.0,0
1,2018,Logistic,Personal Income,0.001348,62.0,0
2,2018,Logistic,Real Disposable Income,0.000814,64.0,0
3,2018,Logistic,Average Hourly Earnings,0.000083,71.0,0
4,2018,Logistic,Weekly Earnings,0.028028,12.0,0
5,2018,Logistic,Wage Growth,0.007096,44.0,0
6,2018,Logistic,Labor Compensation,0.014133,29.0,0
7,2018,Logistic,Self-employment Income,0.000115,70.0,0
8,2018,Logistic,Rental Income,0.000600,67.0,0
9,2018,Logistic,Dividend Income,0.029469,11.0,0


## 25G — Calculate temporal stability

In [25]:
# ============================================================
# STEP 25G — TEMPORAL FEATURE STABILITY
# ============================================================

temporal_stability = (
    walkforward_importance
    .groupby([
        "Model",
        "Feature"
    ])
    .agg(

        Mean_Rank=(
            "Rank",
            "mean"
        ),

        Rank_Std=(
            "Rank",
            "std"
        ),

        Mean_Importance=(
            "Importance",
            "mean"
        ),

        Top10_Count=(
            "Top10",
            "sum"
        ),

        Folds=(
            "Test_Year",
            "nunique"
        )

    )
    .reset_index()
)


temporal_stability[
    "Top10_Frequency"
] = (
    temporal_stability["Top10_Count"]
    /
    temporal_stability["Folds"]
)


display(
    temporal_stability
    .sort_values(
        "Top10_Frequency",
        ascending=False
    )
    .head(30)
)

,Model,Feature,Mean_Rank,Rank_Std,Mean_Importance,Top10_Count,Folds,Top10_Frequency
238,XGBoost,Exchange Rate,2.375,1.407886,0.044534,8,8,1.000
131,Logistic,Savings Rate,2.500,1.511858,0.045289,8,8,1.000
89,Logistic,Economic Policy Uncertainty,2.625,0.916125,0.046416,8,8,1.000
106,Logistic,Inflation Expectations,2.875,2.695896,0.046555,8,8,1.000
157,RandomForest,Credit Card & Revolving Loan Growth,3.000,1.772811,0.032670,8,8,1.000
234,XGBoost,Dividend Income,5.625,2.774244,0.035249,7,8,0.875
88,Logistic,Dividend Income,7.125,2.748376,0.033317,7,8,0.875
19,DecisionTree,Exchange Rate,1.875,1.726888,0.339773,7,8,0.875
136,Logistic,Tax Refunds,7.000,6.611678,0.036813,6,8,0.750
189,RandomForest,Mortgage Rate,9.125,5.221863,0.025306,6,8,0.750


## 25H — Cross-model + cross-time consensus

In [26]:
# ============================================================
# STEP 25H — GLOBAL TEMPORAL CONSENSUS
# ============================================================

global_stability = (
    walkforward_importance
    .groupby("Feature")
    .agg(

        Mean_Rank=(
            "Rank",
            "mean"
        ),

        Rank_Std=(
            "Rank",
            "std"
        ),

        Mean_Importance=(
            "Importance",
            "mean"
        ),

        Top10_Count=(
            "Top10",
            "sum"
        ),

        Total_Evaluations=(
            "Top10",
            "count"
        )
    )
    .reset_index()
)


global_stability[
    "Top10_Frequency"
] = (
    global_stability["Top10_Count"]
    /
    global_stability["Total_Evaluations"]
)


global_stability = (
    global_stability
    .sort_values(
        [
            "Top10_Frequency",
            "Mean_Rank"
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(drop=True)
)


display(
    global_stability.head(20)
)

,Feature,Mean_Rank,Rank_Std,Mean_Importance,Top10_Count,Total_Evaluations,Top10_Frequency
0,Exchange Rate,11.40625,12.080000,0.104998,19,32,0.59375
1,Inflation Expectations,9.06250,10.159812,0.024219,16,32,0.50000
2,Credit Card & Revolving Loan Growth,10.18750,9.268182,0.027792,15,32,0.46875
3,Dividend Income,12.12500,12.950900,0.020756,14,32,0.43750
4,Savings Rate,10.96875,9.930352,0.050048,12,32,0.37500
5,Employment Rate,15.03125,15.241040,0.038430,12,32,0.37500
6,Economic Policy Uncertainty,14.15625,13.695088,0.020298,11,32,0.34375
7,Personal Savings,13.18750,11.745795,0.026992,10,32,0.31250
8,Federal Funds Rate,22.09375,21.691918,0.052872,10,32,0.31250
9,Corporate Bond Yield,11.28125,7.458420,0.018401,9,32,0.28125


## Revised Step 25G — Temporal stability using actual selection

In [27]:
# ============================================================
# STEP 25G — REVISED TEMPORAL FEATURE STABILITY
# ============================================================

# ------------------------------------------------------------
# 1. Feature was actually used by model
# ------------------------------------------------------------

walkforward_importance["Selected"] = (
    walkforward_importance["Importance"] > 0
).astype(int)


# ------------------------------------------------------------
# 2. TRUE Top-10
# ------------------------------------------------------------

walkforward_importance["Top10"] = (
    (
        walkforward_importance["Rank"] <= 10
    )
    &
    (
        walkforward_importance["Importance"] > 0
    )
).astype(int)


# ------------------------------------------------------------
# 3. Temporal stability by model + feature
# ------------------------------------------------------------

temporal_stability = (
    walkforward_importance
    .groupby([
        "Model",
        "Feature"
    ])
    .agg(

        Mean_Importance=(
            "Importance",
            "mean"
        ),

        Importance_Std=(
            "Importance",
            "std"
        ),

        Selected_Count=(
            "Selected",
            "sum"
        ),

        Top10_Count=(
            "Top10",
            "sum"
        ),

        Folds=(
            "Test_Year",
            "nunique"
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# 4. Frequencies
# ------------------------------------------------------------

temporal_stability["Selection_Frequency"] = (
    temporal_stability["Selected_Count"]
    /
    temporal_stability["Folds"]
)


temporal_stability["Top10_Frequency"] = (
    temporal_stability["Top10_Count"]
    /
    temporal_stability["Folds"]
)


# ------------------------------------------------------------
# 5. Sort
# ------------------------------------------------------------

temporal_stability = (
    temporal_stability
    .sort_values(
        [
            "Top10_Frequency",
            "Selection_Frequency",
            "Mean_Importance"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
    .reset_index(drop=True)
)


display(
    temporal_stability.head(30)
)

,Model,Feature,Mean_Importance,Importance_Std,Selected_Count,Top10_Count,Folds,Selection_Frequency,Top10_Frequency
0,Logistic,Inflation Expectations,0.046555,0.009074,8,8,8,1.000,1.000
1,Logistic,Economic Policy Uncertainty,0.046416,0.004904,8,8,8,1.000,1.000
2,Logistic,Savings Rate,0.045289,0.004297,8,8,8,1.000,1.000
3,XGBoost,Exchange Rate,0.044534,0.005341,8,8,8,1.000,1.000
4,RandomForest,Credit Card & Revolving Loan Growth,0.032670,0.005233,8,8,8,1.000,1.000
5,XGBoost,Dividend Income,0.035249,0.005493,8,7,8,1.000,0.875
6,Logistic,Dividend Income,0.033317,0.005045,8,7,8,1.000,0.875
7,DecisionTree,Exchange Rate,0.339773,0.143174,7,7,8,0.875,0.875
8,Logistic,Tax Refunds,0.036813,0.013501,8,6,8,1.000,0.750
9,RandomForest,Mortgage Rate,0.025306,0.006087,8,6,8,1.000,0.750


## 25H — The final cross-model + cross-time table

In [28]:
# ============================================================
# STEP 25H — GLOBAL CROSS-MODEL + CROSS-TIME STABILITY
# ============================================================

global_stability = (
    walkforward_importance
    .groupby("Feature")
    .agg(

        Mean_Importance=(
            "Importance",
            "mean"
        ),

        Importance_Std=(
            "Importance",
            "std"
        ),

        Selected_Count=(
            "Selected",
            "sum"
        ),

        Top10_Count=(
            "Top10",
            "sum"
        ),

        Total_Evaluations=(
            "Top10",
            "count"
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# Frequencies
# ------------------------------------------------------------

global_stability["Selection_Frequency"] = (
    global_stability["Selected_Count"]
    /
    global_stability["Total_Evaluations"]
)


global_stability["Top10_Frequency"] = (
    global_stability["Top10_Count"]
    /
    global_stability["Total_Evaluations"]
)


# ------------------------------------------------------------
# Sort strongest stability first
# ------------------------------------------------------------

global_stability = (
    global_stability
    .sort_values(
        [
            "Top10_Frequency",
            "Selection_Frequency",
            "Mean_Importance"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
    .reset_index(drop=True)
)


display(
    global_stability.head(20)
)

,Feature,Mean_Importance,Importance_Std,Selected_Count,Top10_Count,Total_Evaluations,Selection_Frequency,Top10_Frequency
0,Exchange Rate,0.104998,0.154064,31,19,32,0.96875,0.59375
1,Inflation Expectations,0.024219,0.018580,24,16,32,0.75000,0.50000
2,Credit Card & Revolving Loan Growth,0.027792,0.043153,25,15,32,0.78125,0.46875
3,Dividend Income,0.020756,0.015295,24,14,32,0.75000,0.43750
4,Savings Rate,0.050048,0.086313,27,12,32,0.84375,0.37500
5,Employment Rate,0.038430,0.067967,27,12,32,0.84375,0.37500
6,Economic Policy Uncertainty,0.020298,0.017549,24,11,32,0.75000,0.34375
7,Federal Funds Rate,0.052872,0.119329,28,10,32,0.87500,0.31250
8,Personal Savings,0.026992,0.045752,25,10,32,0.78125,0.31250
9,Unemployment Benefits,0.024426,0.047000,25,9,32,0.78125,0.28125


## 25I — KEY CANDIDATE FEATURE COMPARISON

In [30]:
# ============================================================
# STEP 25I — KEY CANDIDATE FEATURE COMPARISON
# ============================================================

candidate_features = [
    "Exchange Rate",
    "Federal Funds Rate",
    "Delinquency Rate",
    "Dividend Income",
    "Employment Rate",
    "Personal Loan Rate",
    "Vehicle Sales",
    "Full-time Employment"
]


candidate_stability = (
    global_stability[
        global_stability["Feature"]
        .isin(candidate_features)
    ]
    .copy()
    .sort_values(
        "Top10_Frequency",
        ascending=False
    )
)


display(
    candidate_stability.style
    .format({
        "Mean_Importance": "{:.4f}",
        "Importance_Std": "{:.4f}",
        "Selection_Frequency": "{:.1%}",
        "Top10_Frequency": "{:.1%}"
    })
)

,Feature,Mean_Importance,Importance_Std,Selected_Count,Top10_Count,Total_Evaluations,Selection_Frequency,Top10_Frequency
0,Exchange Rate,0.1050,0.1541,31,19,32,96.9%,59.4%
3,Dividend Income,0.0208,0.0153,24,14,32,75.0%,43.8%
5,Employment Rate,0.0384,0.0680,27,12,32,84.4%,37.5%
7,Federal Funds Rate,0.0529,0.1193,28,10,32,87.5%,31.2%
11,Personal Loan Rate,0.0143,0.0123,24,9,32,75.0%,28.1%
12,Vehicle Sales,0.0140,0.0136,22,9,32,68.8%,28.1%
15,Full-time Employment,0.0127,0.0111,22,7,32,68.8%,21.9%
19,Delinquency Rate,0.0134,0.0138,21,5,32,65.6%,15.6%


### interpretation: 

These are our candidate stable predictive features, not yet a final feature-selected dataset.

| Feature                |  Selected | Selection Freq. |    Top-10 | Top-10 Freq. | Assessment         |
| ---------------------- | --------: | --------------: | --------: | -----------: | ------------------ |
| **Exchange Rate**      | **31/32** |       **96.9%** | **19/32** |    **59.4%** | 🟢 Strongest       |
| **Dividend Income**    |     24/32 |           75.0% |     14/32 |    **43.8%** | 🟢 Strong          |
| **Employment Rate**    |     27/32 |       **84.4%** |     12/32 |    **37.5%** | 🟢 Strong          |
| **Federal Funds Rate** |     28/32 |       **87.5%** |     10/32 |        31.2% | 🟡 Moderate/strong |
| Personal Loan Rate     |     24/32 |           75.0% |      9/32 |        28.1% | 🟡 Moderate        |
| Vehicle Sales          |     22/32 |           68.8% |      9/32 |        28.1% | 🟡 Moderate        |
| Full-time Employment   |     22/32 |           68.8% |      7/32 |        21.9% | 🟡 Weaker          |
| Delinquency Rate       |     21/32 |           65.6% |      5/32 |        15.6% | 🟡 Weaker          |


**Strong candidates**\
────────────────────────\
Exchange Rate\
Dividend Income\
Employment Rate\
Federal Funds Rate

**Secondary candidates**\
────────────────────────\
Personal Loan Rate\
Vehicle Sales\
Full-time Employment\
Delinquency Rate